
# 05 Rtree & Spatial Indexing: Why Duplicate-Sale Detection Stays Fast at Scale

Exercise 2 in notebook 01 had you write `check_duplicate_sale()` a function that loops through every
existing parcel and checks `.intersects()` against each one. That works fine for 3 parcels. **It becomes
a real problem once a land banking company has 50,000 parcels in your database**, because checking a new
parcel against every single existing one (a "brute-force" approach) is `O(n)` the time grows linearly
with how much data you have, and at real scale that's the difference between an instant check and a
query that takes seconds.

## The idea behind a spatial index (in plain terms, no math needed)

A spatial index pre-organizes your geometries into a tree structure based on their **bounding boxes**
(the smallest rectangle that contains each shape), grouping nearby shapes together. When you ask "what's
near this point/shape?", the index lets you quickly eliminate huge sections of the data that obviously
can't be relevant, checking only a small candidate set in detail instead of checking everything.

**Rtree is the library that implements this data structure.** You will almost never call Rtree directly
in normal use GeoPandas builds and uses one automatically via `.sindex`. This notebook shows you both
the automatic version and, briefly, what's happening underneath, so the performance benefit isn't a
mystery.

## Proving it with a real timing comparison


In [1]:

import geopandas as gpd
import numpy as np
from shapely.geometry import Point, box
import time

# Generate 5,000 small random "parcels" scattered across an area roughly the size of Buea
np.random.seed(42)
n = 5000
centers_x = np.random.uniform(9.20, 9.30, n)
centers_y = np.random.uniform(4.10, 4.20, n)
size = 0.0005  # small parcel footprint

parcels = gpd.GeoDataFrame({
    "parcel_id": [f"PARC-{i:05d}" for i in range(n)],
    "geometry": [box(x, y, x + size, y + size) for x, y in zip(centers_x, centers_y)],
}, crs="EPSG:4326")

# A new parcel we want to check for duplicate-sale conflicts against all 5,000 existing ones
new_parcel = box(9.2498, 4.1498, 9.2503, 4.1503)

print(f"Existing parcels to check against: {len(parcels)}")


Existing parcels to check against: 5000


In [2]:

# Approach 1: brute force loop through every parcel, check .intersects() one at a time
start = time.perf_counter()
brute_force_matches = [
    pid for pid, geom in zip(parcels["parcel_id"], parcels["geometry"])
    if new_parcel.intersects(geom)
]
brute_force_time = time.perf_counter() - start

print(f"Brute force: found {len(brute_force_matches)} match(es) in {brute_force_time*1000:.2f} ms")


Brute force: found 1 match(es) in 562.90 ms


In [3]:

# Approach 2: using the spatial index .sindex.query() returns candidate positions instantly,
# narrowing 5,000 shapes down to a tiny handful before doing any real geometry check
start = time.perf_counter()
candidate_positions = parcels.sindex.query(new_parcel, predicate="intersects")
indexed_matches = parcels.iloc[candidate_positions]["parcel_id"].tolist()
indexed_time = time.perf_counter() - start

print(f"Indexed:     found {len(indexed_matches)} match(es) in {indexed_time*1000:.2f} ms")
print(f"Speedup: roughly {brute_force_time / max(indexed_time, 1e-9):.0f}x faster")


Indexed:     found 1 match(es) in 10.48 ms
Speedup: roughly 54x faster



The result sets should match the index doesn't change *what* you find, only *how fast* you find it.
Note that `.sindex.query(geometry, predicate="intersects")` already applies the `.intersects()` check
for you and returns exact matches (not just candidates) when you pass a `predicate` this is the
pattern you'll actually use in production code, combining the speed of the index with a one-line call.

## Building your real duplicate-sale checker with the index

Here's `check_duplicate_sale()` from notebook 01, rewritten to use the spatial index this is the
production-grade version of that function.


In [4]:

def check_duplicate_sale_indexed(new_parcel_geom, existing_parcels_gdf):
    """
    Returns the rows of existing_parcels_gdf whose geometry intersects new_parcel_geom,
    using the GeoDataFrame's built-in spatial index for speed at scale.
    """
    candidate_positions = existing_parcels_gdf.sindex.query(new_parcel_geom, predicate="intersects")
    return existing_parcels_gdf.iloc[candidate_positions]

conflicts = check_duplicate_sale_indexed(new_parcel, parcels)
print(f"Found {len(conflicts)} conflicting parcel(s):")
conflicts[["parcel_id"]]


Found 1 conflicting parcel(s):


,parcel_id
1938,PARC-01938



## A brief look under the hood: Rtree directly

You won't usually need this GeoPandas' `.sindex` already gives you an Rtree-backed index for free
but seeing the raw library once demystifies what `.sindex` is actually doing.


In [5]:

from rtree import index

# Build a raw Rtree index by hand, inserting each parcel's bounding box with an integer id
idx = index.Index()
for i, geom in enumerate(parcels["geometry"]):
    idx.insert(i, geom.bounds)  # .bounds gives (minx, miny, maxx, maxy) exactly what Rtree indexes on

# Query it directly: "give me candidate ids whose bounding box intersects this bounding box"
candidate_ids = list(idx.intersection(new_parcel.bounds))
print(f"Rtree found {len(candidate_ids)} bounding-box candidates (before the exact geometry check)")


Rtree found 1 bounding-box candidates (before the exact geometry check)



Notice this raw query returns candidates based on **bounding box** overlap only it's a fast first
pass, not the final answer. That's why `.sindex.query(..., predicate="intersects")` (used above) is
better for real use: it does the fast bounding-box narrowing *and* the precise geometric check in one
call, so you never get a false positive from two bounding boxes overlapping when the actual shapes don't.

## Exercises

### Exercise 1
Increase `n` to 50,000 parcels (rerun the generation cell with a bigger number) and re-time both
approaches. How much bigger does the speed gap get as the dataset grows? This is the exercise that makes
the argument for indexing viscerally obvious rather than theoretical.


In [ ]:
# Your code here try regenerating parcels with n = 50000 and re-running the timing comparison



### Exercise 2
Write a function `find_nearby_parcels(point, parcels_gdf, buffer_degrees)` that returns all parcels
within `buffer_degrees` of a given point, using the spatial index rather than a brute-force loop. (Hint:
buffer the point first, then query the index with the buffered shape same pattern as
`check_duplicate_sale_indexed` above.)


In [7]:
# Your code here


#### Solution

In [6]:

def find_nearby_parcels(point, parcels_gdf, buffer_degrees):
    search_area = point.buffer(buffer_degrees)
    candidate_positions = parcels_gdf.sindex.query(search_area, predicate="intersects")
    return parcels_gdf.iloc[candidate_positions]

nearby = find_nearby_parcels(Point(9.25, 4.15), parcels, 0.002)
print(f"Found {len(nearby)} nearby parcels")


Found 10 nearby parcels



## What's next

You now have fast, correct geometric queries. The next question a real product needs to answer isn't
just "do these two parcels overlap" it's "is high land value clustered in certain areas, and can I use
that pattern to help predict a new parcel's value?" That's a job for spatial *statistics*, not just
spatial *geometry* covered in **`06_spatial_analysis_pysal.ipynb`**.
